# MultiTaxi DQN benchmark

Run from the project's `app/` environment. The benchmark stores final post-training evaluations and convergence checkpoints in a versioned JSON file.

In [ ]:
import json
import os
from dataclasses import dataclass

import numpy as np
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

from scripts import train_dqn_agent, train_dqn_hrm_agent
from src.config import Configuration
from src.models import evaluate_agent


In [ ]:
@dataclass(frozen=True)
class BenchmarkVariant:
    id: str
    name: str
    kind: str
    config: str


@dataclass
class BenchmarkConfiguration(Configuration):
    training_seeds: tuple[int, ...] = tuple(range(42, 52))
    convergence_interval: int = 1000
    rerun_completed: bool = False
    record_version: str = 'dqn-test'
    variants: tuple[BenchmarkVariant, ...] = (
        BenchmarkVariant('dqn', 'DQN', 'dqn', 'dqn.yaml'),
        BenchmarkVariant('dqn_rm', 'DQN + RM', 'dqn', 'dqn_rm.yaml'),
        BenchmarkVariant('dqn_crm', 'DQN + RM + CRM', 'dqn', 'dqn_crm.yaml'),
        BenchmarkVariant('dqn_hrm', 'Deep HRM', 'dqn_hrm', 'dqn_hrm.yaml'),
    )

    @property
    def results_path(self):
        grid = f'{self.multitaxi_grid_size}x{self.multitaxi_grid_size}'
        return os.path.join(self.DATA_PATH, f'multitaxi_{grid}_benchmark_{self.record_version}.json')

    @property
    def checkpoint_episodes(self):
        episodes = list(range(self.convergence_interval, self.n_training_episodes + 1, self.convergence_interval))
        if not episodes or episodes[-1] != self.n_training_episodes:
            episodes.append(self.n_training_episodes)
        return tuple(episodes)

    def config_for(self, seed, variant):
        config = Configuration(yaml_config_path=variant.config)
        config.exp_name = variant.id
        config.multitaxi_grid_size = self.multitaxi_grid_size
        config.video_fps = self.video_fps
        config.n_training_episodes = self.n_training_episodes
        config.n_eval_episodes = self.n_eval_episodes
        run_index = self.variants.index(variant) * len(self.training_seeds) + self.training_seeds.index(seed)
        config.eval_seed_base = (self.eval_seed_base or 0) + run_index
        config.set_seed(seed)
        return config


BENCHMARK = BenchmarkConfiguration(
    n_training_episodes=100000,
    n_eval_episodes=100,
    eval_seed_base=2,
    multitaxi_grid_size=10,
    video_fps=10,
)
RUN_TRAINING = True
print(f'Runs: {len(BENCHMARK.variants) * len(BENCHMARK.training_seeds)}')
print(f'Training episodes per run: {BENCHMARK.n_training_episodes}')
print(f'Evaluation episodes per run: {BENCHMARK.n_eval_episodes}')
print(f'Results: {BENCHMARK.results_path}')


In [ ]:
def normalize_metrics(metrics):
    normalized = {}
    for key, value in metrics.items():
        if isinstance(value, np.generic):
            value = value.item()
        normalized[key] = None if isinstance(value, float) and not np.isfinite(value) else value
    return normalized


def load_runs():
    if not os.path.exists(BENCHMARK.results_path):
        return []
    with open(BENCHMARK.results_path, encoding='utf-8') as file:
        runs = json.load(file)
    if not isinstance(runs, list):
        raise ValueError('Experiment runs must be a list')
    return runs


def save_runs(runs):
    temporary_path = f'{BENCHMARK.results_path}.tmp'
    with open(temporary_path, 'w', encoding='utf-8') as file:
        json.dump(runs, file, indent=2, allow_nan=False)
    os.replace(temporary_path, BENCHMARK.results_path)


def completed_run(run):
    return run['metrics'] is not None and {
        checkpoint['episode'] for checkpoint in run['convergence']
    } == set(BENCHMARK.checkpoint_episodes)


def completed_runs():
    runs = [run for run in load_runs() if completed_run(run)]
    completed_ids = {(run['variant_id'], run['seed']) for run in runs}
    missing = [
        f'{variant.id}:{seed}'
        for variant in BENCHMARK.variants
        for seed in BENCHMARK.training_seeds
        if (variant.id, seed) not in completed_ids
    ]
    if missing:
        raise ValueError(f"Benchmark is incomplete: {', '.join(missing)}")
    return runs


In [ ]:
if RUN_TRAINING:
    runs = load_runs()
    for variant in BENCHMARK.variants:
        for seed in BENCHMARK.training_seeds:
            run_id = (variant.id, seed)
            existing_run = next((run for run in runs if (run['variant_id'], run['seed']) == run_id), None)
            video_path = os.path.join(
                BENCHMARK.VIDEO_PATH,
                f'{BENCHMARK.multitaxi_grid_size}x{BENCHMARK.multitaxi_grid_size}_{variant.id}_seed{seed}_video.gif',
            )
            if existing_run and completed_run(existing_run) and os.path.isfile(video_path) and not BENCHMARK.rerun_completed:
                print(f'Skipping {variant.name}, seed {seed}')
                continue

            config = BENCHMARK.config_for(seed, variant)
            run = {
                'variant_id': variant.id,
                'variant': variant.name,
                'kind': variant.kind,
                'config': variant.config,
                'seed': seed,
                'evaluation_seeds': config.eval_seed,
                'metrics': None,
                'convergence': [],
            }
            runs = [stored_run for stored_run in runs if (stored_run['variant_id'], stored_run['seed']) != run_id]
            runs.append(run)
            save_runs(runs)

            def progress_callback(episode, agent, env, get_propositions):
                if episode % BENCHMARK.convergence_interval and episode != BENCHMARK.n_training_episodes:
                    return
                metrics = normalize_metrics(evaluate_agent(
                    config, agent, get_propositions, env,
                    seeds=run['evaluation_seeds'], report=False, return_metrics=True,
                ))
                run['convergence'].append({'episode': episode, 'metrics': metrics})
                save_runs(runs)
                print(
                    f'{variant.name}, seed {seed}, episode {episode}/{BENCHMARK.n_training_episodes}: '
                    f"reward={metrics['mean_reward']:.2f}, success={metrics['successes']}/{metrics['episodes']}"
                )

            print(f'Starting {variant.name}, seed {seed}: {BENCHMARK.n_training_episodes} training episodes')
            runner = train_dqn_hrm_agent if variant.kind == 'dqn_hrm' else train_dqn_agent
            run['metrics'] = normalize_metrics(runner(config, progress_callback=progress_callback))
            if not any(checkpoint['episode'] == BENCHMARK.n_training_episodes for checkpoint in run['convergence']):
                raise RuntimeError('Training finished without a final convergence checkpoint')
            save_runs(runs)
            print(
                f"{variant.name}, seed {seed}: {run['metrics']['mean_reward']:.2f} reward, "
                f"{run['metrics']['mean_successful_steps']} mean successful steps"
            )


In [ ]:
def summarize_results(runs):
    summary = []
    for variant in BENCHMARK.variants:
        variant_runs = [run for run in runs if run['variant_id'] == variant.id]
        rewards = np.asarray([run['metrics']['mean_reward'] for run in variant_runs])
        steps = np.asarray([
            run['metrics']['mean_successful_steps']
            for run in variant_runs
            if run['metrics']['mean_successful_steps'] is not None
        ])
        summary.append({
            'variant': variant.name,
            'mean_reward': float(rewards.mean()),
            'mean_successful_steps': float(steps.mean()) if len(steps) else None,
            'runs': len(variant_runs),
            'reward_std_across_runs': float(rewards.std()),
            'success_rate': float(np.mean([
                run['metrics']['successes'] / run['metrics']['episodes']
                for run in variant_runs
            ])),
        })
    display(summary)


In [ ]:
def plot_final_evaluation(runs):
    figure = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Final evaluation reward', 'Steps to solve successful episodes'),
    )
    for variant in BENCHMARK.variants:
        variant_runs = [run for run in runs if run['variant_id'] == variant.id]
        rewards = [run['metrics']['mean_reward'] for run in variant_runs]
        steps = [
            run['metrics']['mean_successful_steps']
            for run in variant_runs
            if run['metrics']['mean_successful_steps'] is not None
        ]
        figure.add_trace(go.Box(y=rewards, name=variant.name, boxpoints='all'), row=1, col=1)
        if steps:
            figure.add_trace(go.Box(y=steps, name=variant.name, boxpoints='all', showlegend=False), row=1, col=2)

    figure.update_yaxes(title_text='Mean environment reward', row=1, col=1)
    figure.update_yaxes(title_text='Mean steps', row=1, col=2)
    figure.update_layout(
        title=f'MultiTaxi {BENCHMARK.multitaxi_grid_size}x{BENCHMARK.multitaxi_grid_size} final evaluation',
        template='plotly_white', height=550,
    )
    figure.show()


In [ ]:
def plot_convergence(runs):
    convergence_figure = go.Figure()
    for variant in BENCHMARK.variants:
        values_by_episode = {}
        for run in runs:
            if run['variant_id'] != variant.id:
                continue
            for checkpoint in run['convergence']:
                values_by_episode.setdefault(checkpoint['episode'], []).append(checkpoint['metrics']['mean_reward'])
        episodes = sorted(values_by_episode)
        rewards = [values_by_episode[episode] for episode in episodes]
        convergence_figure.add_trace(go.Scatter(
            x=episodes,
            y=[np.mean(values) for values in rewards],
            error_y={'type': 'data', 'array': [np.std(values) for values in rewards], 'visible': True},
            mode='lines+markers',
            name=variant.name,
        ))

    convergence_figure.update_layout(
        title=f'MultiTaxi {BENCHMARK.multitaxi_grid_size}x{BENCHMARK.multitaxi_grid_size} reward convergence',
        xaxis_title='Training episodes',
        yaxis_title='Mean environment reward',
        template='plotly_white', height=600,
    )
    convergence_figure.show()


## Plot saved results

Set `RUN_TRAINING = True` above only when a new benchmark run is needed. Otherwise this section loads the saved JSON and reuses the plots without retraining.

In [ ]:
runs = completed_runs()
summarize_results(runs)
plot_final_evaluation(runs)
plot_convergence(runs)
